In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [4]:
# Read data/par-to-par.json
with open("data/par-to-par.json", "r") as f:
    data = json.load(f)

In [5]:
df = pd.read_csv("data/clean_data.csv")

for col in ["DATE_FROM", "DATE_TO"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [24]:
rapporteurs = [info["meta"]["advocate_general"] for info in data.values()]
min_length = min(len(r) for r in rapporteurs)
max_length = max(len(r) for r in rapporteurs)
min_length, max_length

(0, 2)

In [26]:
celex_to_advocate_general = {celex: info["meta"]["advocate_general"] for celex, info in data.items()}
celex_to_rapporteur = {celex: info["meta"]["rapporteur"] for celex, info in data.items()}
celex_to_applicant = {celex: info["meta"]["applicant"] for celex, info in data.items()}
celex_to_defendant = {celex: info["meta"]["defendant"] for celex, info in data.items()}

# Add ADVOCATE_TO, ADVOCATE_FROM etc. to df
df["ADVOCATE_FROM"] = df["CELEX_FROM"].apply(lambda x: celex_to_advocate_general.get(x))
df["ADVOCATE_TO"] = df["CELEX_TO"].apply(lambda x: celex_to_advocate_general.get(x))
df["RAPPORT_FROM"] = df["CELEX_FROM"].apply(lambda x: celex_to_rapporteur.get(x))
df["RAPPORT_TO"] = df["CELEX_TO"].apply(lambda x: celex_to_rapporteur.get(x))
df["APPLICANT_FROM"] = df["CELEX_FROM"].apply(lambda x: celex_to_applicant.get(x))
df["APPLICANT_TO"] = df["CELEX_TO"].apply(lambda x: celex_to_applicant.get(x))
df["DEFENDANT_FROM"] = df["CELEX_FROM"].apply(lambda x: celex_to_defendant.get(x))
df["DEFENDANT_TO"] = df["CELEX_TO"].apply(lambda x: celex_to_defendant.get(x))

In [29]:
def normalize(value):
    """Turn values into sets for comparison (list, dict, str, None)."""
    if value is None:
        return set()
    if isinstance(value, list):
        return set(value)
    if isinstance(value, dict):
        return set(value.items())  # dict compared as key-value pairs
    return {value}  # single element

def role_stats(df, role):
    from_col = f"{role}_FROM"
    to_col = f"{role}_TO"
    
    total = 0
    overlap = 0
    identical = 0
    
    for f, t in zip(df[from_col], df[to_col]):
        f_set = normalize(f)
        t_set = normalize(t)
        if not f_set or not t_set:
            continue
        
        total += 1
        if f_set & t_set:
            overlap += 1
        if f_set == t_set:
            identical += 1
    
    return {
        "total": total,
        "overlap": overlap,
        "identical": identical,
        "overlap_rate": overlap / total if total else None,
        "identical_rate": identical / total if total else None,
    }

# Apply to all roles
roles = ["ADVOCATE", "RAPPORT", "APPLICANT", "DEFENDANT"]
stats = {role: role_stats(df, role) for role in roles}

for role, stat in stats.items():
    print(role)
    for key, value in stat.items():
        print(f"{key}: {value}")
    print("\n" + "="*100 + "\n")


ADVOCATE
total: 106437
overlap: 12720
identical: 12705
overlap_rate: 0.11950731418585642
identical_rate: 0.1193663857493165


RAPPORT
total: 110128
overlap: 17906
identical: 17906
overlap_rate: 0.1625926194973122
identical_rate: 0.1625926194973122


APPLICANT
total: 45332
overlap: 31762
identical: 31351
overlap_rate: 0.7006529603811876
identical_rate: 0.6915865172505073


DEFENDANT
total: 46506
overlap: 27035
identical: 25235
overlap_rate: 0.5813228400636477
identical_rate: 0.5426181567969725


